# Diabetic Retinopathy Stage Detection (EfficientNetB0 transfer learning)
Computer Vision coursework. Pipeline: Drive dataset -> exploration -> stratified split -> preprocessing (crop, denoise, CLAHE, edge enhancement) -> augmentation & class imbalance -> transfer learning -> tuning -> training -> evaluation & error analysis -> export for the web app.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Run the cells from top to bottom. If Colab asks to *restart the session* after installing packages, restart and continue with the next cell.

## 1. Setup: clone the repository and install extra packages

In [ ]:
import os
%cd /content
if not os.path.exists("Computer-Vision-CW---diabetic-retinopathy-stage-detection"):
    !git clone https://github.com/ErandiPathirana/Computer-Vision-CW---diabetic-retinopathy-stage-detection.git
%cd /content/Computer-Vision-CW---diabetic-retinopathy-stage-detection
!git pull
# Colab already ships TensorFlow, numpy, pandas, scikit-learn, matplotlib, OpenCV: install only what is missing.
!pip install -q -r requirements-colab.txt

In [ ]:
import sys
sys.path.insert(0, ".")          # so that `from src import ...` works
import tensorflow as tf, numpy as np
from src import config as C
print("TensorFlow:", tf.__version__, "| numpy:", np.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))    # must list one GPU (Runtime > T4 GPU)
C.ensure_dirs()

## 2. Dataset from Google Drive (no Kaggle credentials)
The zip `Images and train file.zip` (APTOS 2019 images + labelled CSV, originally from Kaggle) must be somewhere in your Google Drive. Change `C.DRIVE_ZIP_NAME` in `src/config.py` (or pass `zip_path=`) if your file has a different name.

In [ ]:
from src import data as D
df = D.load_dataset_from_drive()      # mounts Drive, unzips, validates; stops with a clear message on problems
df.head()

## 3. Dataset exploration

In [ ]:
summary = D.explore_dataset(df)       # class distribution, sample images, size/brightness/sharpness statistics
from IPython.display import Image, display
display(Image(str(C.EDA_DIR / "class_distribution.png")))
display(Image(str(C.EDA_DIR / "sample_images.png")))
display(Image(str(C.EDA_DIR / "quality_statistics.png")))
print(open(C.EDA_DIR / "eda_summary.md").read())

## 4. Stratified train / validation / test split (70/15/15)

In [ ]:
train_df, val_df, test_df = D.make_splits(df)
display(Image(str(C.SPLIT_DIR / "split_distribution.png")))

## 5. Preprocessing
Crop black borders -> pad/resize -> denoise -> CLAHE -> unsharp mask. Every image is processed once and cached; the same function is used by the web app.

In [ ]:
from src import preprocess as P
train_df = P.cache_preprocessed(train_df)
val_df   = P.cache_preprocessed(val_df)
test_df  = P.cache_preprocessed(test_df)
P.save_preprocessing_figures(df)
display(Image(str(C.PREPROC_DIR / "preprocessing_steps.png")))
display(Image(str(C.PREPROC_DIR / "preprocessing_metrics.png")))

## 6. Augmentation and class imbalance (training set only)

In [ ]:
from src import augment as A
weights = A.compute_class_weights(train_df)
print("Class weights:", weights)
over_df = A.oversample_minorities(train_df)
A.plot_imbalance_handling(train_df, over_df, weights)
A.save_augmentation_figure(train_df)
check_ds = A.make_dataset(train_df, training=True); A.check_pixel_range(check_ds)
display(Image(str(C.AUG_DIR / "imbalance_handling.png")))
display(Image(str(C.AUG_DIR / "augmentation_examples.png")))

## 7. Model: EfficientNetB0 transfer learning

In [ ]:
from src.model import build_model, configure_phase1, configure_phase2, parameter_counts
m = build_model("efficientnetb0")
configure_phase1(m); print("Phase 1 (head only):", parameter_counts(m))
configure_phase2(m);  print("Phase 2 (fine-tune):", parameter_counts(m))
m.summary(show_trainable=True)
del m

## 8. Experiments on the validation set (optional but recommended for the report)
Backbone comparison, a small hyper-parameter search and the imbalance-strategy comparison. Each run is short. Set `RUN_EXPERIMENTS = False` to skip them and use the defaults from `src/config.py`.

In [ ]:
from src import experiments as X
RUN_EXPERIMENTS = True
best = {}
if RUN_EXPERIMENTS:
    display(X.run_backbone_comparison(train_df, val_df))
    hp = X.run_hparam_search(train_df, val_df); display(hp)
    best = X.best_hparams(hp)
    display(X.run_imbalance_comparison(train_df, val_df))
print("Hyper-parameters used for the final model:", best or "defaults from config.py")

## 9. Final training (two phases, early stopping, LR scheduling, checkpoints)

In [ ]:
from src.train import train_model
model, hist = train_model(train_df, val_df, tag="main", save_final=True, **best)
display(Image(str(C.TRAIN_DIR / "training_curves.png")))

## 10. Evaluation and error analysis (test set)

In [ ]:
import keras
from src.evaluate import evaluate_model
best_model = keras.models.load_model(C.BEST_MODEL_PATH)
metrics = evaluate_model(best_model, test_df)
for f in ["confusion_matrix", "roc_curves", "confidence_histogram", "misclassified_gallery", "gradcam_examples"]:
    display(Image(str(C.EVAL_DIR / f"{f}.png")))
print(open(C.EVAL_DIR / "error_analysis.md").read())

## 11. Export for the web app
Creates `models/` and `webapp/assets/` and a zip to download. Also copy it to Drive so a disconnected session does not lose it.

In [ ]:
from src.export_assets import export_all, export_samples
export_all()
export_samples(test_df)   # one test image per grade for the web app's sample buttons
import shutil, os
!zip -qr export.zip models webapp/assets webapp/samples webapp/samples results/evaluation results/training results/eda results/preprocessing results/augmentation results/experiments results/splits/split_counts.csv
print("Model was trained with TensorFlow", tf.__version__, "- install the same version on your laptop.")
os.makedirs("/content/drive/MyDrive/DR_CW", exist_ok=True)
shutil.copy("export.zip", "/content/drive/MyDrive/DR_CW/export.zip")
from google.colab import files
files.download("export.zip")